# LANGKAH-LANGKAH PRAKTIKUM
## Langkah 1: Persiapan Environment & Pembersihan Dependensi
Buka notebook Google Colab baru, beri nama Praktikum_RAG_GenAI_Permenkes.ipynb, lalu jalankan kode berikut untuk menginstal seluruh dependensi framework RAG yang dibutuhkan (termasuk SDK resmi google-genai):

In [9]:
!pip install -q google-genai chromadb sentence-transformers langchain langchain-community langchain-text-splitters pypdf

In [10]:
# Import main modules
import os
import getpass
import numpy as np
import pandas as pd
import requests

# Import official Google GenAI client module
from google import genai
from google.genai import types

# Import text splitter and vector database components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [11]:
# Download the PDF from the GitHub repo
pdf_url = "https://raw.githubusercontent.com/mpfordreamer/llm-practice/main/permenkes-no-10-tahun-2024.pdf"
pdf_filename = "permenkes_no_10_2024.pdf"

print("Mengunduh dokumen PDF dari GitHub...")
response = requests.get(pdf_url)
with open(pdf_filename, "wb") as f:
    f.write(response.content)
print(f"File {pdf_filename} berhasil diunduh!")

# Load and extract text from the PDF
print("Membaca teks dari PDF...")
loader = PyPDFLoader(pdf_filename)
pages = loader.load()

# Combine all pages into a single text variable (to keep compatibility with your existing code)
text_data = "\n".join([page.page_content for page in pages])

# Initialize recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200       # Overlapping characters between adjacent chunks
)

# Split text document into chunks
chunks = text_splitter.split_text(text_data)

print(f"Total Chunks yang terbentuk dari PDF: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Mengunduh dokumen PDF dari GitHub...
File permenkes_no_10_2024.pdf berhasil diunduh!
Membaca teks dari PDF...
Total Chunks yang terbentuk dari PDF: 16

--- Chunk 1 ---
PERATURAN MENTERI KESEHATAN REPUBLIK INDONESIA 
NOMOR 10 TAHUN 2024 
TENTANG 
JARINGAN DOKUMENTASI DAN INFORMASI HUKUM  
DI LINGKUNGAN KEMENTERIAN KESEHATAN 
 
DENGAN RAHMAT TUHAN YANG MAHA ESA 
 
MENTERI KESEHATAN REPUBLIK INDONESIA, 
 
 
Menimbang : a. bahwa untuk meningkatkan pelayanan kepada 
masyarakat terhadap kebutuhan dokumentasi dan 
informasi hukum bidang kesehatan secara len gkap, 
akurat, mudah, dan cepat perlu pengelolaan jaringan 
dokumentasi dan informasi hukum yang tertata dan 
terselenggara dengan baik;  
  b. bahwa berdasarkan pertimbangan sebagaimana 
dimaksud dalam huruf a dan untuk melaksanakan 
ketentuan Pasal 5 ayat (1) Peraturan Presiden Nomor 33 
Tahun 2012 tentang Jaringan Dokumentasi dan 
Informasi Hukum Nasional, perlu menetapkan Peraturan 
Menteri Kesehatan  tentang Jaringan Dokumentasi dan 


## Langkah 2: Konfigurasi API Key Gemini (Google AI Studio)
SDK terbaru dari Google (google-genai) secara otomatis mencari variabel lingkungan (environment variable) bernama GEMINI_API_KEY untuk melakukan autentikasi ke Google AI Studio.

In [12]:
# Request API Key securely in Google Colab
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Masukkan Google Gemini API Key Anda: ")

print("Koneksi API Key berhasil disiapkan!")

Koneksi API Key berhasil disiapkan!


## Langkah 3: Pemrosesan Dokumen (Document Chunking)
Dokumen regulasi hukum yang panjang harus dipecah menjadi bagian-bagian kecil (chunks) agar pencarian vektor lebih spesifik dan tidak melebihi batasan ukuran input (context window) dari LLM.

In [13]:
# Initialize recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50      # Overlapping characters between adjacent chunks
)

# Split text document into chunks
chunks = text_splitter.split_text(text_data)

print(f"Total Chunks yang terbentuk: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Total Chunks yang terbentuk: 48

--- Chunk 1 ---
PERATURAN MENTERI KESEHATAN REPUBLIK INDONESIA 
NOMOR 10 TAHUN 2024 
TENTANG 
JARINGAN DOKUMENTASI DAN INFORMASI HUKUM  
DI LINGKUNGAN KEMENTERIAN KESEHATAN 
 
DENGAN RAHMAT TUHAN YANG MAHA ESA 
 
MENTERI KESEHATAN REPUBLIK INDONESIA, 
 
 
Menimbang : a. bahwa untuk meningkatkan pelayanan kepada

--- Chunk 2 ---
masyarakat terhadap kebutuhan dokumentasi dan 
informasi hukum bidang kesehatan secara len gkap, 
akurat, mudah, dan cepat perlu pengelolaan jaringan 
dokumentasi dan informasi hukum yang tertata dan 
terselenggara dengan baik;  
  b. bahwa berdasarkan pertimbangan sebagaimana

--- Chunk 3 ---
b. bahwa berdasarkan pertimbangan sebagaimana 
dimaksud dalam huruf a dan untuk melaksanakan 
ketentuan Pasal 5 ayat (1) Peraturan Presiden Nomor 33 
Tahun 2012 tentang Jaringan Dokumentasi dan 
Informasi Hukum Nasional, perlu menetapkan Peraturan


## Langkah 4: Pembuatan Embeddings BAAI/bge-m3 & Penyimpanan ke Database Vektor
Kita akan mengonfigurasi model embedding BAAI/bge-m3 melalui Hugging Face. Model ini menghasilkan representasi vektor dengan akurasi semantik tinggi dalam 1024 dimensi spasial.

In [6]:
# Configure embeddings model BAAI
model_name = "BAAI/bge-m3"

# In Hugging Face Langchain ecosystem device parameter is configured via model kwargs
model_kwargs = {'device': 'cpu'}

# Initialize BAAI embeddings generator
print(f"Mengunduh dan menyiapkan model embeddings ({model_name}) di device: {model_kwargs['device']}...")
embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs={'normalize_embeddings': True} # Normalize embeddings for accurate cosine similarity
)

# Create Vector Store based on ChromaDB
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_db_permenkes" # Persist database to local folder
)

print("Database Vektor ChromaDB (dengan BAAI/bge-m3) berhasil diinisialisasi!")

Mengunduh dan menyiapkan model embeddings (BAAI/bge-m3) di device: cpu...


/tmp/ipykernel_4579/3712713322.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Database Vektor ChromaDB (dengan BAAI/bge-m3) berhasil diinisialisasi!


## Langkah 5: Membangun Pipeline RAG Manual (Native Retriever & Generator)
Di sini, kita tidak lagi menggunakan komponen black-box seperti RetrievalQA dari LangChain. Kita akan menulis fungsi Python secara manual untuk melakukan Retrieve dari ChromaDB, melakukan Augment pada prompt, dan melakukan Generate menggunakan SDK resmi google-genai dengan model Gemini 3.5 Flash.

In [7]:
# Initialize Google GenAI client globally
# Client automatically reads the environment variable GEMINI_API_KEY
client = genai.Client()

def rag_query(query_text):
    # Retrieve relevant documents (Top 10 chunks)
    docs = vector_db.similarity_search(query_text, k=10)
    context = "\n\n".join([doc.page_content for doc in docs])

    # NEW PROMPT: POCLF Framework (Simple & Structured)
    prompt = f"""[Persona]
Anda adalah Ahli Hukum dan Analis Regulasi di Kementerian Kesehatan.

[Objective]
Jawablah pertanyaan pengguna dengan jelas, lengkap, dan akurat berdasarkan dokumen hukum yang diberikan.

[Context]
Dokumen: Permenkes No. 10 Tahun 2024
Isi Dokumen:
{context}

[Limitation]
- HANYA gunakan informasi dari "Isi Dokumen" di atas.
- JANGAN mengarang jawaban (No Hallucination).
- Jika jawaban tidak ada di dalam dokumen, katakan secara jujur bahwa informasi tersebut tidak ditemukan atau terpotong.

[Format]
- Sampaikan jawaban secara langsung, sederhana, dan mudah dipahami.
- Gunakan bullet points (poin-poin) jika terdapat daftar.

Pertanyaan Pengguna: {query_text}
Jawaban:"""

    # Generate response using Gemini AI Studio
    config = types.GenerateContentConfig(
        temperature=0.2,
    )

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=config
    )

    return response.text, context

print("Fungsi RAG Chain Native dengan Google GenAI SDK siap!")

Fungsi RAG Chain Native dengan Google GenAI SDK siap!


## Langkah 6: Pengujian Q&A (Membuktikan Kekuatan RAG Regulasi Kesehatan)
Mari kita uji performa sistem RAG ini dengan mengajukan beberapa pertanyaan medis dan hukum terkait Permenkes No. 10 Tahun 2024 menggunakan fungsi pipeline manual kita.

In [8]:
# Query 1: Information Retrieval (Definisi & Tujuan)
query_1 = "Apa yang dimaksud dengan Jaringan Dokumentasi dan Informasi Hukum (JDIH) Kementerian Kesehatan menurut Permenkes No 10 Tahun 2024?"
print(f"Pertanyaan: {query_1}")

response_1, context_1 = rag_query(query_1)
print("\nJawaban:\n", response_1)


# Query 2: Specific Operational Details (Keanggotaan)
query_2 = "Siapa saja yang berkedudukan sebagai Anggota JDIH Kementerian Kesehatan berdasarkan peraturan ini?"
print(f"Pertanyaan: {query_2}")

response_2, context_2 = rag_query(query_2)
print("\nJawaban:\n", response_2)


# Query 3: Hallucination Test (Sanksi Finansial)
query_3 = "Berapa besaran denda finansial maksimal jika rumah sakit swasta terlambat mengintegrasikan dokumen hukumnya ke dalam sistem JDIH?"
print(f"Pertanyaan: {query_3}")

response_3, context_3 = rag_query(query_3)
print("\nJawaban:\n", response_3)

# Query 4: Specific Article Retrieval (Isi Pasal).
query_4 = "Tolong jelaskan apa saja yang diatur atau disebutkan di dalam Pasal 2 pada Permenkes No 10 Tahun 2024 ini?"
print(f"Pertanyaan: {query_4}")

response_4, context_4 = rag_query(query_4)
print("\nJawaban:\n", response_4)

Pertanyaan: Apa yang dimaksud dengan Jaringan Dokumentasi dan Informasi Hukum (JDIH) Kementerian Kesehatan menurut Permenkes No 10 Tahun 2024?

Jawaban:
 Berdasarkan Permenkes No. 10 Tahun 2024, Jaringan Dokumentasi dan Informasi Hukum Kementerian Kesehatan (JDIH Kemenkes) adalah:

* **Suatu sistem pengelolaan dan pendayagunaan bersama** dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu, dan berkesinambungan.
* **Sarana pemberian pelayanan informasi hukum** secara lengkap, akurat, mudah, dan cepat.
Pertanyaan: Siapa saja yang berkedudukan sebagai Anggota JDIH Kementerian Kesehatan berdasarkan peraturan ini?

Jawaban:
 Berdasarkan dokumen Permenkes No. 10 Tahun 2024 yang Anda berikan, informasi mengenai siapa saja yang secara spesifik berkedudukan sebagai Anggota JDIH Kementerian Kesehatan **tidak dapat ditemukan secara lengkap karena teks dokumen terpotong** pada Pasal 3 ayat (4).

Namun, berdasarkan bagian dokumen yang tersedia, terdapat petunjuk mengenai ha